# Notebook 06: Reusable Inference Pipeline and SHAP Explanations

We package the full workflow into a reusable production-style pipeline:
raw data -> selected features -> trained model -> saved artifacts.


In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import train_test_split

from src.data_loader import load_isolet_dataset
from src.inference_pipeline import FeatureSelectionInferencePipeline, PipelineConfig

warnings.filterwarnings('ignore')
np.random.seed(42)

MODEL_DIR = Path('outputs/models')
METRIC_DIR = Path('outputs/metrics')
FIG_DIR = Path('outputs/figures')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
X, y, metadata = load_isolet_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

config = PipelineConfig(
    var_threshold=0.0,
    corr_threshold=0.95,
    corr_method='spearman',
    mi_k=140,
    rfe_feat=160,
    rfe_step=20,
    rfe_use_cv=False,
    rfe_min_features=40,
    l1_C=1.0,
    shap_k=100,
    random_state=42,
)

pipeline = FeatureSelectionInferencePipeline(config=config)
pipeline.fit(X_fit, y_fit, X_val=X_val, y_val=y_val)

metrics = pipeline.evaluate(X_test, y_test)
metrics


## SHAP global explanation

**Definition:** SHAP values attribute each prediction to additive feature contributions.


In [ ]:
X_test_sel = pipeline.transform(X_test)
background = X_test_sel.sample(min(100, len(X_test_sel)), random_state=42)
explain_samples = X_test_sel.iloc[:150]

explainer = shap.TreeExplainer(pipeline.model, background)
shap_values = explainer.shap_values(explain_samples, check_additivity=False)

if isinstance(shap_values, list):
    sv = shap_values[0] if len(shap_values) == 1 else shap_values[1]
elif getattr(shap_values, 'ndim', 2) == 3:
    sv = shap_values[:, :, 0]
else:
    sv = shap_values

shap.summary_plot(sv, explain_samples, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'pipeline_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
mean_abs_shap = np.abs(sv).mean(axis=0)
shap_rank = pd.DataFrame(
    {
        'feature': explain_samples.columns,
        'mean_abs_shap': mean_abs_shap,
    }
).sort_values('mean_abs_shap', ascending=False)

shap_rank.head(15)


## Artifact persistence and inference demo

**Outputs:**
- `production_model.joblib`
- `selected_features.csv`
- `pipeline_config.json`
- `feature_rankings.json`


In [ ]:
artifacts = pipeline.save_artifacts(MODEL_DIR)

summary_payload = {
    'dataset': metadata,
    'evaluation': metrics,
    'n_original_features': int(X.shape[1]),
    'n_selected_features': int(len(pipeline.selected_features_)),
    'artifacts': artifacts,
}
(METRIC_DIR / 'pipeline_inference_summary.json').write_text(
    json.dumps(summary_payload, indent=2),
    encoding='utf-8',
)

new_samples = X_test.iloc[:5].copy()
preds = pipeline.predict(new_samples)

pd.DataFrame({'sample_index': new_samples.index, 'prediction': preds.values})


## Interpretation

This pipeline is deployment-ready in local environments: deterministic, cache-aware, and artifact-driven.
